# 02 · Análisis exploratorio (EDA)
### Riesgo de acceso a la salud por municipio en Colombia · Fase 1

Parte de los datos que dejan `make clean` y `make features` en `data/processed/`.
Si no existen, ejecute antes el notebook 01 o `make data`.

Las figuras se guardan en `reports/figures/` (no se versionan: se regeneran).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from salud import config
from salud.data.clean import normalizar_texto
from salud.features import CUADRANTES, clasificar_cuadrante

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

fecha = config.ultimo_corte()
leer = lambda nombre: pd.read_parquet(config.PROCESSED_DIR / f"{nombre}_{fecha}.parquet")  # noqa: E731
tabla = leer("tabla_municipal")
bdua_eps = leer("bdua_eps")
bdua_demo = leer("bdua_demo")
tm = tabla.dropna(subset=["afiliados"]).copy()
print(f"Corte {fecha}: {len(tabla):,} municipios, {tm['afiliados'].sum():,.0f} afiliados")

config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8c8b86"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.titleweight": "bold",
        "font.size": 10,
    }
)


def guardar(fig, nombre):
    fig.savefig(config.FIGURES_DIR / f"{nombre}.png", bbox_inches="tight")

In [ ]:
display(tm.describe().T)

pct_int_nac = bdua_eps.loc[bdua_eps["eps_intervenida"], "afiliados"].sum() / bdua_eps["afiliados"].sum()
umbral = config.UMBRAL_DEPENDENCIA_INTERVENIDAS
print(f"% nacional en EPS intervenidas: {pct_int_nac:.1%}")
print(
    f"Municipios con ≥ {umbral:.0%} en EPS intervenidas: {(tm['pct_intervenidas'] >= umbral).sum()} de {len(tm)}"
)

> **Resultado del corte 2026-10-25.** El 42,3 % de los afiliados (21,2 millones) está en una EPS
> intervenida y en 702 de 1.122 municipios (63 %) la mitad o más depende de ellas.
> La mediana municipal es de 10.320 afiliados, 86 % subsidiado, HHI de 5.188 y 2,87
> sedes de IPS por cada 10.000 afiliados.

### 1. ¿Quién concentra la afiliación? Top 15 EPS

In [ ]:
resumen_eps = (
    bdua_eps.groupby(["eps", "eps_intervenida"], as_index=False)["afiliados"]
    .sum()
    .sort_values("afiliados", ascending=False)
)
top = resumen_eps.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(top["eps"], top["afiliados"] / 1e6, color=np.where(top["eps_intervenida"], NARANJA, AZUL), height=0.7)
for y, v in enumerate(top["afiliados"] / 1e6):
    ax.text(
        v + top["afiliados"].max() / 1e6 * 0.01, y, f"{v:.1f} M", va="center", fontsize=8.5, color="#52514e"
    )
ax.set_xlabel("Afiliados (millones)")
ax.set_title("Top 15 EPS por afiliados (contributivo + subsidiado)")
ax.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=NARANJA), plt.Rectangle((0, 0), 1, 1, color=AZUL)],
    labels=["Intervenida", "No intervenida"],
    loc="lower right",
    frameon=False,
)
ax.grid(axis="y", visible=False)
guardar(fig, "01_top_eps")
plt.show()

> **Resultado del corte 2026-10-25.** Nueva EPS tiene casi el doble de afiliados que Sanitas, la
> segunda. De las 15 EPS más grandes, 6 están intervenidas: la crisis incluye a la
> más grande del país.

### 2. Distribución de los indicadores municipales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(tm["pct_intervenidas"] * 100, bins=30, color=AZUL, edgecolor="white")
axes[0].set(title="% afiliados en EPS intervenidas", xlabel="%", ylabel="Municipios")
axes[1].hist(tm["hhi"], bins=30, color=AZUL, edgecolor="white")
axes[1].axvline(config.HHI_MUY_CONCENTRADO, color=GRIS, ls="--", lw=1)
axes[1].text(config.HHI_MUY_CONCENTRADO + 100, axes[1].get_ylim()[1] * 0.9, "muy concentrado", fontsize=8)
axes[1].set(title="Concentración de EPS (HHI)", xlabel="HHI")
axes[2].hist(
    tm["ips_x_10mil"].clip(upper=tm["ips_x_10mil"].quantile(0.99)), bins=30, color=AZUL, edgecolor="white"
)
axes[2].set(title="Sedes de IPS por 10.000 afiliados", xlabel="Sedes / 10.000 (recortado al p99)")
plt.tight_layout()
guardar(fig, "02_distribuciones")
plt.show()

> **Resultado del corte 2026-10-25.** El % en EPS intervenidas tiene forma de "J": en unos 260
> municipios casi el 100 % de la población depende de ellas. Casi todos los
> municipios superan el HHI de 2.500 (muy concentrados) y la mayoría tiene entre 1 y 5
> sedes de IPS por cada 10.000 afiliados.

### 3. Estructura poblacional por régimen

In [ ]:
pir = bdua_demo.groupby(["grupo_edad", "regimen"], observed=True)["afiliados"].sum().unstack() / 1e6
fig, ax = plt.subplots(figsize=(9, 4.2))
x, w = np.arange(len(pir)), 0.4
ax.bar(x - w / 2, pir["CONTRIBUTIVO"], w - 0.03, color=AZUL, label="Contributivo")
ax.bar(x + w / 2, pir["SUBSIDIADO"], w - 0.03, color=NARANJA, label="Subsidiado")
ax.set_xticks(x, pir.index.astype(str), fontsize=8.5)
ax.set(ylabel="Afiliados (millones)", title="Afiliados por grupo de edad y régimen")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
guardar(fig, "03_edad_regimen")
plt.show()

> **Resultado del corte 2026-10-25.** El grupo más grande en ambos regímenes es el de 19 a 45 años
> (unos 10 millones cada uno). El subsidiado tiene 4,3 millones de niños de 5 a 15
> años frente a 2,5 millones en el contributivo: si una EPS del subsidiado falla,
> afecta a muchos niños.

### 4. Relación entre indicadores (Spearman)

In [ ]:
vars_corr = [
    "pct_intervenidas",
    "hhi",
    "pct_eps_principal",
    "n_eps",
    "ips_x_10mil",
    "pct_rural",
    "pct_subsidiado",
]
corr = tm[vars_corr].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(vars_corr)), vars_corr, rotation=45, ha="right")
ax.set_yticks(range(len(vars_corr)), vars_corr)
for i in range(len(vars_corr)):
    for j in range(len(vars_corr)):
        ax.text(
            j,
            i,
            f"{corr.iat[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if abs(corr.iat[i, j]) > 0.6 else "#0b0b0b",
        )
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman")
ax.set_title("Correlación entre indicadores municipales")
guardar(fig, "04_correlaciones")
plt.show()

> **Resultado del corte 2026-10-25.** La dependencia de EPS intervenidas casi no se relaciona con la
> oferta de IPS (0,02): son dos dimensiones distintas y el índice de riesgo necesita
> ambas. El HHI y el % de la EPS principal miden casi lo mismo (0,98): en el
> clustering conviene usar solo uno. Los municipios más rurales son más subsidiados
> (0,59) y tienen menos EPS (−0,53).

### 5. El cruce clave: dependencia de EPS intervenidas vs. oferta de IPS
Cada punto es un municipio y su tamaño es proporcional a los afiliados. El
cuadrante superior izquierdo (**alta dependencia y baja oferta**) es donde está el
riesgo. La clasificación (`salud.features.clasificar_cuadrante`) es exploratoria: el
índice final se construye en la fase 2.

Se usa escala logarítmica en el eje horizontal (todos los municipios tienen al menos
una sede, así que no hay ceros). Esto corrige los valores negativos que mostraba la
escala `symlog` de la primera versión.

In [ ]:
tm["cuadrante"] = clasificar_cuadrante(tm)
mediana_oferta = tm["ips_x_10mil"].median()
riesgo = tm["cuadrante"] == CUADRANTES[0]
tam = np.sqrt(tm["afiliados"]) / 8

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    tm.loc[~riesgo, "ips_x_10mil"],
    tm.loc[~riesgo, "pct_intervenidas"] * 100,
    s=tam[~riesgo],
    color=AZUL,
    alpha=0.45,
    edgecolor="white",
    linewidth=0.5,
    label="Resto de municipios",
)
ax.scatter(
    tm.loc[riesgo, "ips_x_10mil"],
    tm.loc[riesgo, "pct_intervenidas"] * 100,
    s=tam[riesgo],
    color=NARANJA,
    alpha=0.8,
    edgecolor="white",
    linewidth=0.5,
    label=CUADRANTES[0],
)
ax.axhline(config.UMBRAL_DEPENDENCIA_INTERVENIDAS * 100, color=GRIS, ls="--", lw=1)
ax.axvline(mediana_oferta, color=GRIS, ls="--", lw=1)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))  # 1, 10, 100 en vez de 10¹
ax.set(
    xlabel="Sedes de IPS por 10.000 afiliados (escala log)",
    ylabel="% afiliados en EPS intervenidas",
    title="Dependencia de EPS intervenidas vs. oferta de IPS por municipio",
)
for _, r in tm[riesgo].nlargest(8, "afiliados").iterrows():
    ax.annotate(
        r["nom_mpio"].title(),
        (r["ips_x_10mil"], r["pct_intervenidas"] * 100),
        fontsize=7.5,
        xytext=(4, 4),
        textcoords="offset points",
    )
ax.legend(frameon=True, framealpha=0.9, loc="best", markerscale=0.6)
guardar(fig, "05_dependencia_vs_oferta")
plt.show()

tm.groupby("cuadrante").agg(municipios=("cod_mpio", "size"), afiliados=("afiliados", "sum")).reindex(
    CUADRANTES
)

> **Resultado del corte 2026-10-25.** 
>
> | Grupo | Municipios | Afiliados |
> |---|---|---|
> | **Alta dependencia · baja oferta (riesgo)** | **362** | **6.627.358** |
> | Alta dependencia · oferta media/alta | 340 | 9.704.525 |
> | Baja dependencia · baja oferta | 199 | 6.690.539 |
> | Baja dependencia · oferta media/alta | 221 | 27.111.599 |
>
> 362 municipios con 6,6 millones de personas (13 % de los afiliados) están en la
> zona de mayor riesgo. La mediana nacional de oferta es 2,87 sedes por 10.000.

### 6. Mapa exploratorio

In [ ]:
import plotly.express as px

mapa = tm.dropna(subset=["latitud", "longitud"]).assign(
    pct_int=lambda d: (d["pct_intervenidas"] * 100).round(1)
)
kw = dict(
    lat="latitud",
    lon="longitud",
    color="pct_int",
    size="afiliados",
    size_max=28,
    hover_name="nom_mpio",
    hover_data={
        "dpto": True,
        "eps_principal": True,
        "sedes_ips": True,
        "ips_x_10mil": ":.1f",
        "latitud": False,
        "longitud": False,
    },
    color_continuous_scale="Oranges",
    range_color=(0, 100),
    zoom=4.3,
    center={"lat": 4.6, "lon": -74.1},
    height=650,
    labels={"pct_int": "% en EPS intervenidas"},
    title="% de afiliados en EPS intervenidas por municipio (tamaño = afiliados)",
)
try:
    fig = px.scatter_map(mapa, map_style="carto-positron", **kw)
except AttributeError:  # versiones antiguas de plotly
    fig = px.scatter_mapbox(mapa, mapbox_style="carto-positron", **kw)
fig.show()

> **Resultado del corte 2026-10-25.** Mayor dependencia (promedio ponderado por afiliados): Guaviare
> y Guainía (100 %), Caquetá (90 %), Arauca (85 %), Vaupés (84 %), Putumayo (79 %).
> Menor: La Guajira (14 %), Antioquia (20 %) y Bogotá (22 %).

### 7. Ranking preliminar y zoom departamental

In [ ]:
cols_rank = [
    "dpto",
    "nom_mpio",
    "afiliados",
    "eps_principal",
    "pct_eps_principal",
    "pct_intervenidas",
    "sedes_ips",
    "ips_x_10mil",
    "pct_rural",
]
print("Municipios en el cuadrante de riesgo, por población afiliada (Top 20 nacional):")
display(tm[riesgo].nlargest(20, "afiliados")[cols_rank])

if config.DEPTO_FOCO:
    foco = tm[tm["dpto"].map(normalizar_texto) == normalizar_texto(config.DEPTO_FOCO)]
    if len(foco):
        pct = foco["pct_intervenidas"].mul(foco["afiliados"]).sum() / foco["afiliados"].sum()
        print(
            f"\n{config.DEPTO_FOCO}: {len(foco)} municipios · {foco['afiliados'].sum():,.0f} afiliados · "
            f"{pct:.1%} en EPS intervenidas"
        )
        display(
            foco["eps_principal"].value_counts().rename("municipios donde es la EPS principal").to_frame()
        )
        display(
            foco.sort_values(["pct_intervenidas", "ips_x_10mil"], ascending=[False, True]).head(15)[cols_rank]
        )

> **Resultado del corte 2026-10-25.** Los municipios de mayor riesgo con más población son Jamundí,
> Girón y Villa del Rosario; en 12 de los 20 primeros la EPS principal es Nueva EPS.
> Muchos están junto a una ciudad grande cuyas IPS probablemente usan, algo que el
> indicador actual no mide.
>
> En Antioquia solo el 19,9 % está en EPS intervenidas, porque Savia Salud (EPS
> principal en 73 de 125 municipios) no se marcó como intervenida. El riesgo aparece
> en el Bajo Cauca y el Nordeste, que dependen de Coosalud: Nechí (94 % y 0,71 sedes
> por 10.000), El Bagre y Cáceres. Para el escenario con Savia Salud intervenida:
> `INCLUIR_SAVIA=1 make clean features`.

## Hallazgos y limitaciones

**Hallazgos**
- 42,3 % de los afiliados del país está en una EPS intervenida; en 702 de 1.122 municipios es la mitad o más.
- 362 municipios con 6,6 millones de afiliados combinan alta dependencia y baja oferta de IPS.
- Ningún municipio está totalmente sin IPS, pero la mediana es de 3 sedes por municipio.
- La afiliación está muy concentrada (mediana del HHI 5.188).
- Dependencia y oferta no están correlacionadas: el índice de riesgo debe combinar ambas.
- El riesgo se concentra en la Orinoquía, la Amazonía, el sur del país y municipios intermedios junto a grandes ciudades.

**Limitaciones**
1. **Datos agregados.** La BDUA no tiene registros por paciente: el análisis es municipal.
2. **Oferta aproximada.** El REPS cuenta sedes, no capacidad (camas, especialistas), ni qué IPS contrata con qué EPS.
3. **Municipios vecinos.** Solo se cuentan las IPS dentro del municipio.
4. **Cortes distintos.** BDUA de octubre de 2026 y REPS de marzo de 2026.
5. **La lista de EPS intervenidas cambia.** Es un parámetro de `config.py` que se revisa en cada corte; la decisión sobre Savia Salud cambia por completo Antioquia.
6. **Sin histórico en el portal.** Para ver tendencias hay que guardar cada corte (`data/raw/<fecha>/`).